# Bulgarian TTS — Demo (phoneme FastSpeech 2 + fine-tuned HiFi-GAN)

This notebook presents the trained Bulgarian text-to-speech model end to end:

> **text → phonemes → FastSpeech 2 → HiFi-GAN → audio**

It loads the **latest phoneme checkpoint** and the **fine-tuned HiFi-GAN
generator**, synthesizes speech for a sentence, and plays it inline. Run the
cells from top to bottom.

## 1. Configuration

In [ ]:
# Latest checkpoint is auto-selected from this directory (files named N.pth.tar).
CKPT_DIR = "output/ckpt/Bulgarian"

# Fine-tuned Bulgarian HiFi-GAN generator (use g_*, not the do_* discriminator).
FINETUNED_VOCODER = "hifigan_finetune/g_00080000"

# Where the wav/png are written.
RESULT_DIR = "local_inference_results"

# Text to synthesize.
TEXT = "Здравейте! Това е демонстрация на български синтез на реч."

# Prosody controls.
DURATION_CONTROL = 1.15   # larger = slower speech
PITCH_CONTROL    = 1.0
ENERGY_CONTROL   = 1.0

# MFA command for out-of-lexicon (OOV) words. Words already in the runtime
# lexicon need no MFA; leave as "mfa" if Montreal Forced Aligner is installed.
MFA_CMD = "mfa"

## 2. Setup and imports

In [ ]:
import os, re, sys
from pathlib import Path

# Resolve the repository root (this notebook lives in notebooks/).
REPO_ROOT = Path.cwd()
if (REPO_ROOT / "synthesize.py").exists():
    pass
elif (REPO_ROOT.parent / "synthesize.py").exists():
    REPO_ROOT = REPO_ROOT.parent
else:
    raise RuntimeError("Run this notebook from inside the repository checkout.")

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import numpy as np
import torch
import yaml

import synthesize as synthesize_module
from synthesize import preprocess_bulgarian
from utils.model import get_model
import hifigan
from bg_text_normalizer import normalize as contextual_normalize
from bulgarian_normalization import normalize_with_punctuation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Load the Bulgarian configuration.
with open("config/Bulgarian/preprocess.yaml", encoding="utf-8") as f:
    preprocess_config = yaml.safe_load(f)
with open("config/Bulgarian/model.yaml", encoding="utf-8") as f:
    model_config = yaml.safe_load(f)
with open("config/Bulgarian/train.yaml", encoding="utf-8") as f:
    train_config = yaml.safe_load(f)

RESULT_DIR = Path(RESULT_DIR)
RESULT_DIR.mkdir(parents=True, exist_ok=True)


def latest_step(ckpt_dir):
    steps = []
    for p in Path(ckpt_dir).glob("*.pth.tar"):
        m = re.fullmatch(r"(\d+)\.pth\.tar", p.name)
        if m:
            steps.append(int(m.group(1)))
    if not steps:
        raise FileNotFoundError(f"No N.pth.tar checkpoints in {ckpt_dir}")
    return max(steps)


STEP = latest_step(CKPT_DIR)
print("using checkpoint step:", STEP)

## 3. Load FastSpeech 2 (latest phoneme checkpoint)

In [ ]:
class _Args:
    restore_step = STEP


# Point the checkpoint loader at the chosen directory.
train_config["path"]["ckpt_path"] = CKPT_DIR

configs = (preprocess_config, model_config, train_config)
model = get_model(_Args(), configs, device, train=False)
print("FastSpeech2 loaded;", sum(p.numel() for p in model.parameters()), "parameters")

## 4. Load the fine-tuned HiFi-GAN vocoder

In [ ]:
with open("hifigan/config.json") as f:
    vocoder_config = hifigan.AttrDict(json.load(f))

vocoder = hifigan.Generator(vocoder_config)
ckpt = torch.load(FINETUNED_VOCODER, map_location=device, weights_only=False)
assert "generator" in ckpt, (
    f"{FINETUNED_VOCODER} is not a generator checkpoint (needs key 'generator'; "
    "use g_*, not do_*)."
)
vocoder.load_state_dict(ckpt["generator"])
vocoder.eval()
vocoder.remove_weight_norm()
vocoder.to(device)
print("fine-tuned HiFi-GAN loaded from", FINETUNED_VOCODER)

## 5. Text → phonemes

The raw text is normalized (numbers, dates, currencies, punctuation) and then
converted to MFA phones via the Bulgarian runtime lexicon. Words outside the
lexicon fall back to MFA G2P (requires MFA installed).

In [ ]:
raw_text = TEXT
normalized_text = normalize_with_punctuation(contextual_normalize(raw_text))

print("Raw text:       ", raw_text)
print("Normalized text:", normalized_text)
print()

# preprocess_bulgarian prints the phone sequence and returns the phone ids.
phone_ids = preprocess_bulgarian(
    normalized_text,
    preprocess_config,
    mfa_cmd=MFA_CMD,
)
print()
print("phoneme tokens:", len(phone_ids))

## 6. FastSpeech 2 → mel → HiFi-GAN → audio

In [ ]:
from IPython.display import Audio, Image, display

output_id = "demo"
batchs = [(
    [output_id],            # ids
    [raw_text],             # raw_texts
    np.array([0]),          # speakers
    np.array([phone_ids]),  # phone-id sequences
    np.array([len(phone_ids)]),
    len(phone_ids),
)]
control_values = (PITCH_CONTROL, ENERGY_CONTROL, DURATION_CONTROL)

train_config["path"]["result_path"] = str(RESULT_DIR)
configs = (preprocess_config, model_config, train_config)

# FastSpeech2 forward pass + HiFi-GAN vocoder; writes <output_id>.wav and .png.
synthesize_module.synthesize(model, STEP, configs, vocoder, batchs, control_values)

wav_path = RESULT_DIR / f"{output_id}.wav"
png_path = RESULT_DIR / f"{output_id}.png"
print("wav:", wav_path)
print("png:", png_path)

display(Image(filename=str(png_path)))
display(Audio(filename=str(wav_path)))

## Notes

- To use the **universal** HiFi-GAN instead of the fine-tuned one, replace the
  vocoder cell with:
  `from utils.model import get_vocoder; vocoder = get_vocoder(model_config, device)`.
- Change `TEXT` and re-run cells 5 and 6 to synthesize another sentence.
- `DURATION_CONTROL` adjusts speaking rate (larger = slower); `PITCH_CONTROL`
  and `ENERGY_CONTROL` scale pitch and loudness.
- Out-of-lexicon words require Montreal Forced Aligner with the `bulgarian_mfa`
  G2P model; in-lexicon words need nothing extra.